# imports

In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score
import mlflow
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
import os

# config

In [10]:
ASSET = "BTC"
INTERVAL = "1h"

# load data 

In [11]:
train_df = pd.read_parquet('../../../data/processed/train_btc_1h.parquet')
test_df = pd.read_parquet('../../../data/processed/test_btc_1h.parquet')

# targets nd tests

In [12]:
y_train = train_df.pop('target_direction')
X_train = train_df.drop(columns=['target_1h'], errors='ignore')
y_test = test_df.pop('target_direction')
X_test = test_df.drop(columns=['target_1h'], errors='ignore')

# mlflow

In [13]:
mlflow.set_tracking_uri("http://localhost:5000")
experiment_name = f"{ASSET}_{INTERVAL}_XGBoost"
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(disable=True)

# before tuning up getting important features 

In [14]:
# importances = xgb_model.feature_importances_
# feature_names = X_train.columns
# indices = np.argsort(importances)[::-1]
# print("15 important features:")
# for i in range(15):
#     print(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}")

In [15]:
# plt.figure(figsize=(10, 6))
# plt.title("xgboost imp features")
# plt.bar(range(15), importances[indices][:15], align="center")
# plt.xticks(range(15), [feature_names[i] for i in indices[:15]], rotation=45, ha='right')
# plt.tight_layout()
# plt.show()

# run with mlflow

In [16]:
with mlflow.start_run(run_name="XGBoost_Run4_Tuned_GridSearch"):
    param_grid = {
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [100, 200],
        'subsample': [0.8, 1.0]
    }
    
    base_xgb = xgb.XGBClassifier(
        tree_method='hist', 
        device='cuda', 
        eval_metric='logloss', 
        random_state=42
    )
    
    grid_search = GridSearchCV(
        estimator=base_xgb,
        param_grid=param_grid,
        scoring='accuracy',
        cv=3,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print("\n best params found:")
    print(grid_search.best_params_)
    
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("best_test_accuracy", test_acc)
    
    print(f"\n best model test accuracy: {test_acc:.4f} ")
    
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
    model_dir = os.path.join(project_root, 'src', 'models')
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, f'{ASSET}_{INTERVAL}_xgboost_model.json')
    best_model.save_model(model_path)
    print(f"Model saved to: {model_path}")
    
mlflow.xgboost.autolog(disable=False)

Fitting 3 folds for each of 36 candidates, totalling 108 fits

 best params found:
{'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}

 best model test accuracy: 0.5260 
Model saved to: c:\Users\Manindra\Desktop\Financial_Data_Analyzer\src\models\BTC_1h_xgboost_model.json


2026/05/23 13:58:19 WARNING mlflow.utils.autologging_utils: MLflow xgboost autologging is known to be compatible with 2.1.0 <= xgboost, but the installed version is 2.0.3. If you encounter errors during autologging, try upgrading / downgrading xgboost to a compatible version, or try upgrading MLflow.


🏃 View run XGBoost_Run4_Tuned_GridSearch at: http://localhost:5000/#/experiments/14/runs/9b06cc3678bd464fb71ff5c804241078
🧪 View experiment at: http://localhost:5000/#/experiments/14
